In [1]:
import numpy as np
import os

# 把这里换成你的文件路径
FILE_PATH = "E:/Project/AMR-Benchmark/L20250425_103838_F6000000000_S400000_B100000_M0_IQ_ID0_nozaoshen_100ksps_QPSK.dat"

def diagnose(file_path):
    print(f"正在诊断文件: {os.path.basename(file_path)}")
    
    # 读取前 1000 个字节用于分析
    try:
        raw_bytes = np.fromfile(file_path, dtype=np.uint8, count=1000)
    except FileNotFoundError:
        print("错误：找不到文件，请检查路径。")
        return

    print("-" * 60)
    
    # 假设 1: 小端序 int16 (Little Endian) - 最常见
    data_le = raw_bytes.view(np.int16)
    print(f"[假设 A: int16 小端序 (默认)]")
    print(f"前10个数值: {data_le[:10]}")
    print(f"最大值: {np.max(data_le)}, 最小值: {np.min(data_le)}")
    print(f"是否有很多 0? : {np.sum(data_le == 0)} / {len(data_le)}")
    
    # 检查步长 (看看数值是不是 0, 256, 512 这种大跳跃)
    diffs = np.abs(np.diff(data_le[:100].astype(int)))
    diffs = diffs[diffs > 0] # 去掉0
    if len(diffs) > 0:
        min_step = np.min(diffs)
        print(f"**非零最小跳变 (步长)**: {min_step}")
        if min_step >= 256:
            print("  -> 警报：步长很大，这极可能是【大端序】数据！")
    print("-" * 60)

    # 假设 2: 大端序 int16 (Big Endian)
    data_be = raw_bytes.view(np.dtype('>i2'))
    print(f"[假设 B: int16 大端序 (Big Endian)]")
    print(f"前10个数值: {data_be[:10]}")
    print(f"最大值: {np.max(data_be)}, 最小值: {np.min(data_be)}")
    print("-" * 60)

    # 假设 3: float32 (complex64 的一部分)
    # 如果文件是 float32，int16 读出来会是巨大的乱码
    try:
        data_float = raw_bytes.view(np.float32)
        print(f"[假设 C: float32]")
        print(f"前10个数值: {data_float[:10]}")
        print(f"数值是否在 [-1, 1] 之间? : {np.all(np.abs(data_float[:20]) <= 1.5)}")
    except:
        pass
    print("-" * 60)

if __name__ == "__main__":
    diagnose(FILE_PATH)

正在诊断文件: L20250425_103838_F6000000000_S400000_B100000_M0_IQ_ID0_nozaoshen_100ksps_QPSK.dat
------------------------------------------------------------
[假设 A: int16 小端序 (默认)]
前10个数值: [0 0 0 0 0 0 0 0 0 0]
最大值: 50, 最小值: -52
是否有很多 0? : 59 / 500
**非零最小跳变 (步长)**: 2
------------------------------------------------------------
[假设 B: int16 大端序 (Big Endian)]
前10个数值: [0 0 0 0 0 0 0 0 0 0]
最大值: 12800, 最小值: -13057
------------------------------------------------------------
[假设 C: float32]
前10个数值: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
数值是否在 [-1, 1] 之间? : True
------------------------------------------------------------
